In [3]:
import itertools

def hamming_distance(s1, s2):
    """计算两个等长字符串的汉明距离（不匹配的字符数）"""
    return sum(ch1 != ch2 for ch1, ch2 in zip(s1, s2))

def generate_d_neighbors(pattern, d):
    """生成与pattern汉明距离不超过d的所有可能k-mer"""
    nucleotides = ['A', 'C', 'G', 'T']
    neighbors = set()
    k = len(pattern)
    
    # 遍历所有可能的错配位置组合
    for num_mismatches in range(0, d + 1):
        # 选择要替换的位置
        for positions in itertools.combinations(range(k), num_mismatches):
            # 生成替换字符的所有组合
            for replacements in itertools.product(nucleotides, repeat=num_mismatches):
                # 构建新序列
                new_pattern = list(pattern)
                for idx, pos in enumerate(positions):
                    new_pattern[pos] = replacements[idx]
                neighbors.add(''.join(new_pattern))
    return neighbors

def MotifEnumeration(Dna, k, d):
    """实现模体枚举算法"""
    Patterns = set()
    
    # 遍历每个DNA序列
    for sequence in Dna:
        # 遍历当前序列中的每个k-mer
        for i in range(len(sequence) - k + 1):
            pattern = sequence[i:i+k]
            # 生成当前k-mer的所有d-近邻
            neighbors = generate_d_neighbors(pattern, d)
            
            # 检查每个近邻是否在所有DNA序列中出现
            for candidate in neighbors:
                valid = True
                for seq in Dna:
                    found = False
                    # 在序列中搜索候选模体
                    for j in range(len(seq) - k + 1):
                        substring = seq[j:j+k]
                        if hamming_distance(candidate, substring) <= d:
                            found = True
                            break
                    if not found:
                        valid = False
                        break
                if valid:
                    Patterns.add(candidate)
    
    return sorted(Patterns)  # 按字母顺序排序返回

In [6]:
def HammingDist(p, q):
    """
    Calculate the Hamming distance between two sequences p and q.
    
    The Hamming distance between two strings is the number of positions at which the corresponding symbols are different.
    
    Parameters:
    p (sequence): The first sequence.
    q (sequence): The second sequence.
    
    Returns:
    int: The Hamming distance between p and q.
    
    Note:
    The lengths of the two sequences do not need to be the same. The function compares elements at corresponding positions until the end of the shorter sequence is reached.
    """
    # Use the zip function to pair up elements from both sequences and generate a list of booleans indicating whether the elements are unequal
    # Sum the booleans (True values count as 1) to calculate the total number of differences, which represents the Hamming distance
    return sum(p != q for p, q in zip(p, q))

def ImmediateNeighbor(Pattern):
    """
    Finds all immediate neighbor patterns for a given pattern.

    Parameters:
    Pattern (str): The input pattern string.

    Returns:
    list: A list containing all immediate neighbor patterns.
    """
    # Initialize the neighborhood list with the input pattern itself
    Neighborhood = [Pattern]
    # Iterate over each position in the pattern
    for i in range(len(Pattern)):
        # Iterate over all nucleotides
        for j in ["A", "C", "G", "T"]:
            # If the current nucleotide is not equal to j, replace it and add to the neighborhood list
            if Pattern[i] != j:
                Neighborhood.append(Pattern[:i] + j + Pattern[i+1:])
    return Neighborhood

def Neighbors(Pattern, d):
    """
    Finds all neighbors of a given pattern within a specified Hamming distance.

    Parameters:
    Pattern (str): The input pattern string.
    d (int): Maximum Hamming distance.

    Returns:
    list: A list containing all patterns that are within the maximum Hamming distance d of the input pattern.
    """
    # Initialize the neighborhood with the input pattern itself
    Neighborhood = [Pattern]
    # For each distance from 0 to d, find the immediate neighbors of the current patterns
    for j in range(d):
        # For each current neighbor, find its immediate neighbors and add them to the neighborhood
        for Neighbor in Neighborhood:
            Neighborhood = Neighborhood + ImmediateNeighbor(Neighbor)
        # Remove duplicate neighbors to ensure uniqueness in the neighborhood
        Neighborhood = list(set(Neighborhood))
    # Remove any remaining duplicates before returning the final neighborhood
    Neighborhood = list(set(Neighborhood))
    return Neighborhood

def MultiNeighbors(k, d, Dna):
    """
    Calculate the d-neighborhood set of k-mers for each DNA sequence.

    Parameters:
    k -- int, the value of k in k-mers
    d -- int, the maximum distance for the neighborhood
    Dna -- list, a list of input DNA sequences

    Returns:
    Neighborhood -- dict, the d-neighborhood set of k-mers for each DNA sequence
    """
    # Initialize an empty dictionary to store the neighborhood sets for each DNA sequence
    Neighborhood = {}
    # Iterate over each DNA sequence in the input
    for String in Dna:
        # Initialize an empty list for the current DNA sequence's neighborhood set
        Neighborhood[String] = []
        # Iterate over all possible k-mers in the current DNA sequence
        for i in range(len(String) - k + 1):
            # Extract the k-mer at the current position
            Pattern = String[i:i+k]
            # Add the d-neighborhood of the current k-mer to the neighborhood set
            Neighborhood[String] = Neighborhood[String] + Neighbors(Pattern, d)
        # Remove duplicates from the neighborhood set
        Neighborhood[String] = list(set(Neighborhood[String]))
    # Return the neighborhood sets for each DNA sequence
    return Neighborhood

def MotifEnumeration(Dna, k, d):
    """
    Main function: enumerates all common motifs in DNA sequences given certain conditions.

    Parameters:
    Dna - A list containing multiple DNA sequences.
    k - The length of the motif.
    d - The allowed Hamming distance.

    Returns:
    A list containing all motifs that satisfy the conditions.
    """
    # Initialize the motifs list
    Motifs = []
    # Build a dictionary of k-mers and their neighborhoods for each DNA sequence
    Neighborhood = MultiNeighbors(k, d, Dna)
    
    # Iterate through each DNA sequence
    for String in Dna:
        # Iterate through all possible k-mers and their neighborhoods for the current DNA sequence
        for Pattern in Neighborhood[String]:
            # Ensure the current pattern has not been added to the motifs list
            if Pattern not in Motifs:
                # Initialize a counter to count the number of matching DNA sequences
                Count = 0
                # Iterate through each DNA sequence
                for String2 in Dna:
                    # Iterate through all possible k-mers for the current DNA sequence
                    for i in range(len(String2) - k + 1):
                        # If the Hamming distance between the current pattern and the k-mer is no more than d, increment the counter
                        if HammingDist(Pattern, String2[i:i+k]) <= d:
                            Count += 1
                            break
                # If all DNA sequences match, add the current pattern to the motifs list
                if Count == len(Dna):
                    Motifs.append(Pattern)
    
    # Remove duplicate motifs
    Motifs = list(set(Motifs))
    # Return all motifs that satisfy the conditions
    return(Motifs)

# Open the specified text file and prepare to read its data
with open("dataset_30302_8.txt", "r") as f:
    # Read the first line of the file, split it into two parts, and convert them to integers.
    # It is assumed that the first line contains two integers separated by a space, representing k and d.
    k, d = f.readline().split()
    k, d = int(k), int(d)
    # Read the second line of the file and split it into a list of strings.
    # It is assumed that the second line contains DNA sequences separated by spaces.
    Dna = f.readline().split()
    # Call the MotifEnumeration function with the list of DNA sequences, k, and d, to obtain a set of motifs that meet the criteria.
    Motifs = MotifEnumeration(Dna, k, d)
    # Convert the obtained list of motifs into a string separated by spaces and print it out.
    print(k, d)
    print(Dna)
    print(" ".join(Motifs))

5 1
['CTTGATGTGGTCGGTACTATACATA', 'CGTATTAGATGGGTTTGTGATCGAA', 'CAGGCTGTGAATGGCCCTGGCGTCG', 'GCGCATGTGCAGGCGCGTGAAGTCT', 'TGTGTATCACTCGAACGAAGTTAAA', 'GTACTGCGATACCGAAACCATGTGT']
TGTGA GTGTG TGTGG GATGT TGTGC TGTGT


In [1]:
def countMotifPercent(motifs):
    count = {}
    columns = []
    for i in range(len(motifs[0])):
        columns.append([motif[i] for motif in motifs])
    for i in range(len(columns)):
        count[i] = {'A': columns[i].count('A')/len(columns[i]), 'C': columns[i].count('C')/len(columns[i]), 'G': columns[i].count('G')/len(columns[i]), 'T': columns[i].count('T')/len(columns[i])}

    return count





import math
def motifEntropy(motifs):
    entropy = 0
    percents = countMotifPercent(motifs)
    for i in range(len(percents)):
        for nucleotide in percents[i]:
            if percents[i][nucleotide] != 0:
                entropy += percents[i][nucleotide] * math.log2(percents[i][nucleotide])
    return -entropy

In [2]:
motifEntropy([

    "TCGGGGGTTTTT",

    "CCGGTGACTTAC",

    "ACGGGGATTTTC",

    "TTGGGGACTTTT",

    "AAGGGGACTTCC",

    "TTGGGGACTTCC",

    "TCGGGGATTCAT",

    "TCGGGGATTCCT",

    "TAGGGGAACTAC",

    "TCGGGTATAACC"

])

9.916290005356972

In [3]:
# Function to compute the Hamming distance
def hamming_distance(s1, s2):
    return sum(1 for a, b in zip(s1, s2) if a != b)

# Function to find motifs matching a pattern in Dna
def motifs(pattern, dna):
    k = len(pattern)
    motifs = []
    
    for string in dna:
        min_distance = float('inf')
        best_kmer = ""
        for i in range(len(string) - k + 1):
            kmer = string[i:i + k]
            distance = hamming_distance(pattern, kmer)
            if distance < min_distance:
                min_distance = distance
                best_kmer = kmer
        motifs.append(best_kmer)
    
    return motifs

# Example Input
pattern = "GATTACA"
dna = [
    "CGTACGATTACACGAT",
    "GATTACAGTCAGATTG",
    "GATCGATTGACGATTC",
    "CGATTACACGTACGAT"
]

# Find motifs
result = motifs(pattern, dna)
print(result)

['GATTACA', 'GATTACA', 'GATTGAC', 'GATTACA']


In [15]:
def hamming_distance(s1, s2):
    """计算两个等长字符串的汉明距离"""
    return sum(1 for a, b in zip(s1, s2) if a != b)

def total_distance(pattern, dna_list):
    """计算Pattern与DNA序列集合的总距离"""
    k = len(pattern)
    total = 0
    for sequence in dna_list:
        min_dist = float('inf')
        # 在每条序列中扫描所有k-mer
        for i in range(len(sequence) - k + 1):
            kmer = sequence[i:i+k]
            dist = hamming_distance(pattern, kmer)
            if dist < min_dist:
                min_dist = dist
        total += min_dist
    return total

def median_string(dna_input, k):
    """寻找DNA序列集合的中位串，支持多种输入格式"""
    from itertools import product
    
    # 处理输入格式：如果输入是字符串则分割为列表
    if isinstance(dna_input, str):
        dna_list = dna_input.split()
    elif isinstance(dna_input, list):
        dna_list = dna_input
    else:
        raise ValueError("输入格式错误：请提供字符串或列表")
    
    min_distance = float('inf')
    best_pattern = None
    bases = ['A', 'C', 'G', 'T']
    
    # 生成所有可能的k-mer组合
    for pattern_tuple in product(bases, repeat=k):
        pattern = ''.join(pattern_tuple)
        dist = total_distance(pattern, dna_list)
        
        # 更新最小距离和最佳模式
        if dist < min_distance:
            min_distance = dist
            best_pattern = pattern
    
    return best_pattern

# 使用示例
if __name__ == "__main__":
    # 用户提供的DNA序列（空格分隔的字符串）
    dna_input = "CCGCGGATCAGATTCACAGGAGCGAATATAGAACCCTTCTAT TAGCAGTTCTACTTCCGCATGTAGGTTCGAGTGCGTTGTCGT TTCTACCTATTTCCACGAAGGGCCAACCCTTAGACTCCCTTA TTCTACTGGCACGTTCACCGGTTTTTTGCCTAACCGATCGGG GGTGTGTCCTTTCGCCTGTTCTACGCTGAGCAAGAAAATAGA CTATCATGAAGCGCTTTGAAGGCATTCTATAAATATTCCCCG CGGAACACCTCATCGACATGATTTGAATACTTCTAAGGAAGG TTCTAGAGACCGACCGTAGGCACAGGTTACGTACCCGACCTC TTCTATATCATTCAGAGCCAATTCACATAGGGTTGTAAGATA GGGAGCAAACGAACTGCCTAGCCTTTCTATCCACCAACTAAC"
    k_value = 6
    
    # 直接调用函数处理
    result = median_string(dna_input, k_value)
    print(f"DNA序列：{dna_input}")
    print(f"最佳中位串 (k={k_value})：{result}")

DNA序列：CCGCGGATCAGATTCACAGGAGCGAATATAGAACCCTTCTAT TAGCAGTTCTACTTCCGCATGTAGGTTCGAGTGCGTTGTCGT TTCTACCTATTTCCACGAAGGGCCAACCCTTAGACTCCCTTA TTCTACTGGCACGTTCACCGGTTTTTTGCCTAACCGATCGGG GGTGTGTCCTTTCGCCTGTTCTACGCTGAGCAAGAAAATAGA CTATCATGAAGCGCTTTGAAGGCATTCTATAAATATTCCCCG CGGAACACCTCATCGACATGATTTGAATACTTCTAAGGAAGG TTCTAGAGACCGACCGTAGGCACAGGTTACGTACCCGACCTC TTCTATATCATTCAGAGCCAATTCACATAGGGTTGTAAGATA GGGAGCAAACGAACTGCCTAGCCTTTCTATCCACCAACTAAC
最佳中位串 (k=6)：TTCTAC


In [11]:
def Dist(Pattern, Text):
    """
    Calculate the minimum Hamming distance between a given pattern and text.

    Parameters:
    Pattern (str): The pattern to match.
    Text (str): The text to search the pattern in.

    Returns:
    int: The minimum Hamming distance between the pattern and text.
    """
    # Initialize a list to store the Hamming distances at each position
    Dist = []
    # Iterate through the text, calculating the Hamming distance of substrings of the same length as the pattern
    for i in range(len(Text) - len(Pattern) + 1):
        # Append the calculated Hamming distance to the list
        Dist.append(HammingDist(Pattern, Text[i:i+len(Pattern)]))
    # Return the minimum value in the list, which is the minimum Hamming distance
    return min(Dist)

def Dist2(k, Dna):
    """
    Calculate the minimum sum of distances between each k-mer and all other sequences in the given list of DNA sequences,
    and return the k-mer with the smallest distance sum.

    Parameters:
    k (int) -- The length of the k-mer.
    Dna (list) -- A list of DNA sequences.

    Returns:
    key (str) -- The k-mer with the smallest distance sum.
    """
    # Initialize a dictionary to store the sum of distances for each k-mer
    dist2 = {}
    
    # Iterate through each DNA sequence
    for i in range(len(Dna)):
        # Iterate through all possible k-mers in the current DNA sequence
        for j in range(len(Dna[i]) - k + 1):
            # Extract the current k-mer
            Pattern = Dna[i][j:j+k]
            # Initialize the distance sum to 0
            dist = 0
            # Iterate through all DNA sequences, calculate the distance between the current k-mer and each sequence,
            # and add it to the distance sum
            for l in range(len(Dna)):
                dist += Dist(Pattern, Dna[l])
            # Add the current k-mer and its distance sum to the dictionary
            dist2[Pattern] = dist
    
    # Find the minimum distance sum in the dictionary
    Min = min(dist2.values())
    # Iterate through all k-mers in the dictionary to find the one with the smallest distance sum
    for key in dist2.keys():
        if dist2[key] == Min:
            # Return the k-mer with the smallest distance sum
            return key

In [7]:
table_text = """Rv1738                       +     57  1.94e-09 CGGCTCAGTT AGGGCCGGAAGTCCCCAATG TGGCAGACCT
Rv1737c                      -    139  1.94e-09 CGGCTCAGTT AGGGCCGGAAGTCCCCAATG TGGCAGACCT
Rv1733c                      +    107  8.27e-09 GCAGGCGGGA AGGGGCGAAAGTCCCTTATC CGACAATAAA
Rv2032                       -    172  9.45e-09 GGCAGACAAC AGGGTCAATGGTCCCCAAGT GGATCACCGA
Rv2031c                      +    114  9.45e-09 GGCAGACAAC AGGGTCAATGGTCCCCAAGT GGATCACCGA
Rv3134c                      -    136  3.71e-08 TCCCCACCTG AGGGCCGTTAGTCCTTATCG ACCGATGACA
Rv1997                       -    159  6.48e-08 GGTGTGGATC AGGGCCGTAAGTCATCCACT GCAGGGACCT
Rv3131                       +    143  9.90e-08 CCGTTGAGGC AGGGCCAAAGGTCCTCCGCT GGCGAGCCTT
Rv3130c                      -    155  9.90e-08 CCGTTGAGGC AGGGCCAAAGGTCCTCCGCT GGCGAGCCTT
Rv2006                       +    186  9.90e-08 TCACCGGTCA TGAGGCTTTAGTCCCCAATC GGACGGCCAA
Rv2005c                      -    178  9.90e-08 TCACCGGTCA TGAGGCTTTAGTCCCCAATC GGACGGCCAA
Rv2627c                      +    200  1.34e-07 TGACTTCCGG CGGGACGTAAGTCCCTAACG CGTCGTTCCG
Rv2623                       -    118  1.48e-07 CACTAGCCCA ACGGGCTTAGGTCCTCAATG GATCACGAAT
Rv0574c                      -    137  1.48e-07 CCTCGTGGGT GGGGCCGAAGGTCCTCAAGA CCGCGCCCAA
Rv1996                       -    173  2.89e-07 GATCGCGTCA GGAGGCGATGGTCCCTAACC CAAGGGCATT
Rv0571c                      +    201  6.33e-07 CAGTCGGCCC AGGGACCTAATTCCATATTT GGAGCCATGA
Rv0079                       -    160  6.33e-07 GGACGCGGAC AGGGCCGAAAGTCCCATGCC AAATGTGGGC
Rv2626c                      -     62  6.87e-07 AGGCCACCTC AAAGTCGAAGGTTGTCAATT GATTTCACCA
Rv2628                       +    216  1.11e-06 GGGCCAGTGA AGTGACGAAAGACCCCAGTG GACACGGACT
Rv2007c                      -    165  1.49e-06 GGAAAGGACC AAAGGCCTTATTCGTCAAGT TCGTCACTGG
Rv2631                       -     45  5.92e-06 CGACCCTGCG AGAACCGAAGTTCGTTGAGT GACACCGCCT
Rv1734c                      +    204  1.08e-05 CGGGGGCTCT GGGGACCGAAGTCCCCGGGC TCGGAGCTAT"""

In [34]:
table_text_2 = """Rv2032                       +    866  6.68e-10 CGTGGCCAGG GCTAGGGACAGAAGTCCCCG AAGCGCGGGC
Rv2031c                      -    920  6.68e-10 CGTGGCCAGG GCTAGGGACAGAAGTCCCCG AAGCGCGGGC
Rv1738                       +    804  5.39e-09 CGCCGGCTCA GTTAGGGCCGGAAGTCCCCA ATGTGGCAGA
Rv1737c                      -    892  5.39e-09 CGCCGGCTCA GTTAGGGCCGGAAGTCCCCA ATGTGGCAGA
Rv2629                       +     15  1.51e-08 GGAACGACGC GTTAGGGACTTACGTCCCGC CGGAAGTCAC
Rv2628                       +    724  1.51e-08 GGAACGACGC GTTAGGGACTTACGTCCCGC CGGAAGTCAC
Rv2627c                      -    949  1.51e-08 GGAACGACGC GTTAGGGACTTACGTCCCGC CGGAAGTCAC
Rv2006                       -    935  4.31e-08 TGGCCGTCCG ATTGGGGACTAAAGCCTCAT GACCGGTGAC
Rv2005c                      +    929  4.31e-08 TGGCCGTCCG ATTGGGGACTAAAGCCTCAT GACCGGTGAC
Rv1813c                      -    730  5.49e-08 ACCTGACCCG GGTAGGGCATAAAGTCTCTA ACAGCACCGA
Rv2623                       +    891  1.22e-07 AGCCCGTTGG GCTAGTGACAAACGCCTCCT GAGCAAAACC
Rv1734c                      +    951  1.35e-07 AGTCGGGGGC TCTGGGGACCGAAGTCCCCG GGCTCGGAGC
Rv1733c                      +    422  1.35e-07 AGTCGGGGGC TCTGGGGACCGAAGTCCCCG GGCTCGGAGC
Rv3134c                      +    908  1.67e-07 ACGGCCCTCA GGTGGGGACCAACGCCCCTG GGAGATAGCG
Rv3133c                      +     74  1.67e-07 ACGGCCCTCA GGTGGGGACCAACGCCCCTG GGAGATAGCG
Rv0079                       -    913  1.67e-07 CACGGACGCG GACAGGGCCGAAAGTCCCAT GCCAAATGTG
Rv1996                       +    924  2.75e-07 ATGCCCTTGG GTTAGGGACCATCGCCTCCT GACGCGATCG
Rv0574c                      -    890  3.03e-07 CGGCCTCGTG GGTGGGGCCGAAGGTCCTCA AGACCGCGCC
Rv3131                       +    890  4.39e-07 GAGCCGTTGA GGCAGGGCCAAAGGTCCTCC GCTGGCGAGC
Rv3130c                      -    908  4.39e-07 GAGCCGTTGA GGCAGGGCCAAAGGTCCTCC GCTGGCGAGC
Rv0571c                      -    950  1.10e-06 CATGGCTCCA AATATGGAATTAGGTCCCTG GGCCGACTGA
Rv0570                       -    615  1.72e-06 GCCACCAGCG CCGGTGGCCAAAAGTCCCCT CGGGACCCGC
Rv0569                       -    907  1.72e-06 GCCACCAGCG CCGGTGGCCAAAAGTCCCCT CGGGACCCGC
Rv1997                       -    912  1.99e-06 GCCGGTGTGG ATCAGGGCCGTAAGTCATCC ACTGCAGGGA
Rv2007c                      +    923  5.04e-06 GAACTTGACG AATAAGGCCTTTGGTCCTTT CCGGTAGGGG
Rv0572c                      +    513  5.62e-06 CGTCGTCGGG GCGAAGGCCAAACGACCCTA GAACAACAAG
Rv3841                       +     13  7.64e-06 TCGAATCCGT AGTGGTGACCGACGCCACCG GCGCCGAACC"""

In [35]:
import re

pattern = r" ([ATCG]{20}) "
result = re.findall(pattern, table_text_2)

In [36]:
with open('output_2.txt', 'w') as file:
    file.write('\n'.join(result))

In [24]:
import re

pattern = r'[+-]\s{4,5}(\d+)'
match_strings = re.findall(pattern, table_text_2)

numbers = []
for s in match_strings:
    try:
        if '.' in s:
            numbers.append(float(s))  # 小数转float
        else:
            numbers.append(int(s))    # 整数转int
    except ValueError:
        # 处理异常情况（如空字符串，实际很少出现）
        pass

# 输出结果
print("提取的数字字符串：", match_strings)
print("转换后的数字列表：", numbers)

#57, 139, 107, 172, 114, 136, 159, 143, 155, 186, 178, 200, 118, 137, 173, 201, 160, 62, 216, 165, 45, 204
#56, 140, 106, 118, 168, 199, 215, 137, 185, 179, 119, 138, 160, 142, 156, 161, 174, 200, 173, 203, 162

提取的数字字符串： ['56', '140', '106', '118', '168', '199', '215', '137', '185', '179', '119', '138', '160', '142', '156', '161', '174', '200', '173', '203', '162']
转换后的数字列表： [56, 140, 106, 118, 168, 199, 215, 137, 185, 179, 119, 138, 160, 142, 156, 161, 174, 200, 173, 203, 162]
